In [2]:
!git clone https://github.com/TanvirLimon12/PCMMD-FL-TL-KD /kaggle/working/PCMMD-repo
%cd /kaggle/working/PCMMD-repo/PCMMD
!pip install -q -r requirements.txt

Cloning into '/kaggle/working/PCMMD-repo'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 80 (delta 26), reused 77 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 276.59 KiB | 17.29 MiB/s, done.
Resolving deltas: 100% (26/26), done.
[Errno 2] No such file or directory: '/kaggle/working/PCMMD-repo/PCMMD'
/kaggle/working
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


In [4]:
%cd /kaggle/working/PCMMD-repo
!ls
!pip install -q -r requirements.txt

/kaggle/working/PCMMD-repo
analyze_clients.py  evaluate.py      requirements.txt	      train_fewshot.py
checkpoints	    figures	     results		      train_kd.py
configs		    fl		     statistical_analysis.py  utils
data		    INSTRUCTIONS.md  train_centralized.py
docs		    models	     train_fedavg.py
error_analysis.py   README.md	     train_fedprox.py


In [5]:
!pip install -q -r requirements.txt

In [6]:
!ls /kaggle/input

datasets


In [7]:
import glob, pathlib, collections, yaml, os
import pandas as pd

fold_dir = str(pathlib.Path('data/eda').resolve())
assert (pathlib.Path(fold_dir) / 'fold_1.csv').exists(), "data/eda/fold_1.csv missing in repo!"
chk = pd.read_csv(pathlib.Path(fold_dir) / 'fold_1.csv')
print("fold_dir label check:", chk['label'].unique(), "| rows:", len(chk))

# image_root guess না করে পুরো /kaggle/input ব্যবহার করছি (basename index পুরো tree জুড়ে খুঁজবে)
image_root = '/kaggle/input'

# sanity check: যেই ফাইলটায় আগে আটকেছিল সেটা আদৌ কোথাও আছে কিনা
test_hits = glob.glob('/kaggle/input/**/IMG_20240608_104340_crop4.jpg', recursive=True)
print("test file found at:", test_hits)
assert test_hits, "এই basename পুরো /kaggle/input এর কোথাও নেই — dataset-এ ফাইলটা মিসিং হতে পারে!"

print("fold_dir   =", fold_dir)
print("image_root =", image_root)

for cfg in glob.glob('configs/*.yaml'):
    d = yaml.safe_load(open(cfg))
    d['data_root'] = fold_dir; d['fold_dir'] = fold_dir; d['image_root'] = image_root
    yaml.safe_dump(d, open(cfg, 'w'), sort_keys=False)
print("patched:", glob.glob('configs/*.yaml'))

from data.folds import validate_fold
print(validate_fold(fold_dir, 1))

fold_dir label check: ['plasma' 'non_plasma'] | rows: 2026
test file found at: ['/kaggle/input/datasets/falaqueabrar69/eda-dataset/all_outputs/kaggle/working/patient_cells/08/non_plasma/IMG_20240608_104340_crop4.jpg', '/kaggle/input/datasets/falaqueabrar69/eda-dataset/patient_cells/08/non_plasma/IMG_20240608_104340_crop4.jpg']
fold_dir   = /kaggle/working/PCMMD-repo/data/eda
image_root = /kaggle/input
patched: ['configs/centralized.yaml', 'configs/fedavg.yaml', 'configs/fedprox.yaml', 'configs/kd.yaml', 'configs/fewshot.yaml']
{'fold_id': 1, 'n_train': 1624, 'n_test': 402, 'train_plasma': 361, 'train_non_plasma': 1263, 'test_plasma': 98, 'test_non_plasma': 304, 'n_patients': 10, 'n_train_clients': 6, 'n_val_patients': 2, 'val_patients': ['10', '5'], 'patient_leakage': []}


In [8]:

import glob, pandas as pd

hits = glob.glob('/kaggle/input/**/fold_1.csv', recursive=True)
print("Found fold_1.csv at:")
for h in hits:
    df = pd.read_csv(h)
    print("\n", h)
    print(" columns:", list(df.columns))
    print(" label values:", df['label'].unique() if 'label' in df.columns else "NO label col")
    print(" rows:", len(df))

Found fold_1.csv at:

 /kaggle/input/datasets/falaqueabrar69/eda-dataset/all_outputs/kaggle/working/pcmmd_eda_outputs/folds/fold_1.csv
 columns: ['path', 'patient_id', 'patient_diagnosis', 'label', 'image_type', 'set_name', 'fold', 'role', 'md5']
 label values: ['detection_unlabeled']
 rows: 344

 /kaggle/input/datasets/falaqueabrar69/eda-dataset/pcmmd_eda_outputs/folds/fold_1.csv
 columns: ['path', 'patient_id', 'patient_diagnosis', 'label', 'image_type', 'set_name', 'fold', 'role', 'md5']
 label values: ['detection_unlabeled']
 rows: 344


In [9]:
from data.folds import validate_fold
print(validate_fold(fold_dir, 1))

{'fold_id': 1, 'n_train': 1624, 'n_test': 402, 'train_plasma': 361, 'train_non_plasma': 1263, 'test_plasma': 98, 'test_non_plasma': 304, 'n_patients': 10, 'n_train_clients': 6, 'n_val_patients': 2, 'val_patients': ['10', '5'], 'patient_leakage': []}


In [15]:
!python train_centralized.py --config configs/centralized.yaml --backbone resnet50 --folds 1

09:28:48 | INFO | Centralized CV | backbone=resnet50 folds=[1] loss=weighted_ce ft=full | device=cuda
09:28:48 | INFO | Fold 1: train=1624 test=402 | clients=6 val_patients=2 (['1', '10'])
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 162MB/s]
09:29:11 | INFO |   [resnet50 f1] ep 001/30 loss=0.1068 val_f1=0.9405 val_acc=0.9728 val_auc=0.9965
09:29:26 | INFO |   [resnet50 f1] ep 002/30 loss=0.0286 val_f1=0.8555 val_acc=0.9383 val_auc=0.9963
09:29:40 | INFO |   [resnet50 f1] ep 003/30 loss=0.0208 val_f1=0.9794 val_acc=0.9901 val_auc=0.9977
09:29:55 | INFO |   [resnet50 f1] ep 004/30 loss=0.0195 val_f1=0.9697 val_acc=0.9852 val_auc=0.9973
09:30:10 | INFO |   [resnet50 f1] ep 005/30 loss=0.0137 val_f1=0.9794 val_acc=0.9901 val_auc=0.9974
09:30:26 | INFO |   [resnet50 f1] ep 006/30 loss=0.0222 val_f1=0.9634 val_acc=0.9827 val_auc=0.99

In [9]:
!python train_centralized.py --config configs/centralized.yaml --backbone resnet50

11:33:37 | INFO | Centralized CV | backbone=resnet50 folds=[1, 2, 3, 4, 5] loss=weighted_ce ft=full | device=cuda
11:33:37 | INFO | Fold 1: train=1624 test=402 | clients=6 val_patients=2 (['10', '3'])
11:33:58 | INFO |   [resnet50 f1] ep 001/30 loss=0.1083 val_f1=0.9497 val_acc=0.9777 val_auc=0.9937
11:34:12 | INFO |   [resnet50 f1] ep 002/30 loss=0.0319 val_f1=0.9399 val_acc=0.9728 val_auc=0.9929
11:34:27 | INFO |   [resnet50 f1] ep 003/30 loss=0.0395 val_f1=0.9503 val_acc=0.9777 val_auc=0.9920
11:34:42 | INFO |   [resnet50 f1] ep 004/30 loss=0.0234 val_f1=0.9457 val_acc=0.9752 val_auc=0.9938
11:34:57 | INFO |   [resnet50 f1] ep 005/30 loss=0.0235 val_f1=0.9492 val_acc=0.9777 val_auc=0.9926
11:35:12 | INFO |   [resnet50 f1] ep 006/30 loss=0.0047 val_f1=0.9560 val_acc=0.9802 val_auc=0.9938
11:35:29 | INFO |   [resnet50 f1] ep 007/30 loss=0.0055 val_f1=0.9451 val_acc=0.9752 val_auc=0.9940
11:35:46 | INFO |   [resnet50 f1] ep 008/30 loss=0.0054 val_f1=0.9560 val_acc=0.9802 val_auc=0.9944

In [10]:
!python train_centralized.py --config configs/centralized.yaml --backbone efficientnet_b0

11:58:30 | INFO | Centralized CV | backbone=efficientnet_b0 folds=[1, 2, 3, 4, 5] loss=weighted_ce ft=full | device=cuda
11:58:30 | INFO | Fold 1: train=1624 test=402 | clients=6 val_patients=2 (['3', '7'])
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 148MB/s]
11:58:52 | INFO |   [efficientnet_b0 f1] ep 001/30 loss=0.3118 val_f1=0.8465 val_acc=0.9187 val_auc=0.9933
11:58:59 | INFO |   [efficientnet_b0 f1] ep 002/30 loss=0.0909 val_f1=0.9146 val_acc=0.9581 val_auc=0.9940
11:59:06 | INFO |   [efficientnet_b0 f1] ep 003/30 loss=0.0508 val_f1=0.9100 val_acc=0.9557 val_auc=0.9946
11:59:14 | INFO |   [efficientnet_b0 f1] ep 004/30 loss=0.0342 val_f1=0.9479 val_acc=0.9754 val_auc=0.9946
11:59:21 | INFO |   [efficientnet_b0 f1] ep 005/30 loss=0.0295 val_f1=0.9733 val_acc=0.9877 val_auc=0.9950
11:59:28 |

In [11]:
!python train_centralized.py --config configs/centralized.yaml --backbone mobilenet_v3

12:12:27 | INFO | Centralized CV | backbone=mobilenet_v3 folds=[1, 2, 3, 4, 5] loss=weighted_ce ft=full | device=cuda
12:12:27 | INFO | Fold 1: train=1624 test=402 | clients=6 val_patients=2 (['3', '7'])
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100%|███████████████████████████████████████| 9.83M/9.83M [00:00<00:00, 102MB/s]
12:12:46 | INFO |   [mobilenet_v3 f1] ep 001/30 loss=0.2742 val_f1=0.6354 val_acc=0.7512 val_auc=0.9109
12:12:51 | INFO |   [mobilenet_v3 f1] ep 002/30 loss=0.0769 val_f1=0.8764 val_acc=0.9458 val_auc=0.9813
12:12:56 | INFO |   [mobilenet_v3 f1] ep 003/30 loss=0.0377 val_f1=0.9312 val_acc=0.9680 val_auc=0.9882
12:13:00 | INFO |   [mobilenet_v3 f1] ep 004/30 loss=0.0457 val_f1=0.9278 val_acc=0.9655 val_auc=0.9898
12:13:05 | INFO |   [mobilenet_v3 f1] ep 005/30 loss=0.0206 val_f1=0.9326 val_acc=0.9680 val_auc=0.9913
12:13:10 | INFO |   [mobilenet_v3 f1] ep 0

In [13]:
!python error_analysis.py --config configs/centralized.yaml \
    --weights checkpoints/centralized/efficientnet_b0_fold1.pth --fold 1 --tag effnet --backbone efficientnet_b0 --panel

TP=96 TN=303 FP=1 FN=2 (plasma false-negatives=2)
Saved: results/error_effnet_fold1_misclassified.csv


In [14]:
!python train_fewshot.py --config configs/fewshot.yaml

12:26:23 | INFO | Few-shot | backbones=['efficientnet_b0', 'mobilenet_v3'] pcts=[5, 10, 20, 50, 100] folds=[1, 2, 3, 4, 5] | device=cuda
12:26:23 | WARNING | fewshot_5 fold1: dropped 17 rows colliding with test (anti-leakage).
12:27:02 | INFO |   [efficientnet_b0 p5 f1] best_val_f1=0.8920
12:27:37 | INFO |   [mobilenet_v3 p5 f1] best_val_f1=0.6174
12:27:38 | WARNING | fewshot_10 fold1: dropped 39 rows colliding with test (anti-leakage).
12:28:27 | INFO |   [efficientnet_b0 p10 f1] best_val_f1=0.9694
12:29:09 | INFO |   [mobilenet_v3 p10 f1] best_val_f1=0.9479
12:29:11 | WARNING | fewshot_20 fold1: dropped 86 rows colliding with test (anti-leakage).
12:30:00 | INFO |   [efficientnet_b0 p20 f1] best_val_f1=0.9846
12:30:53 | INFO |   [mobilenet_v3 p20 f1] best_val_f1=0.9412
12:30:54 | WARNING | fewshot_50 fold1: dropped 216 rows colliding with test (anti-leakage).
12:32:36 | INFO |   [efficientnet_b0 p50 f1] best_val_f1=0.9897
12:33:58 | INFO |   [mobilenet_v3 p50 f1] best_val_f1=0.9897
1

In [10]:
%%writefile utils/common.py
"""
utils/common.py
---------------
Reproducibility, device, logging, config snapshots and model-complexity tooling
(parameter count, on-disk size, FLOPs/MACs, inference latency).

These are shared by every train/eval script so behaviour is identical everywhere.
"""
from __future__ import annotations

import json
import logging
import os
import random
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import torch
import torch.nn as nn
import yaml

SEED = 42  # project-wide fixed seed — DO NOT change

_CONFIG_DEFAULTS: Dict[str, Any] = {
    "data_root": "./",
    "fold_dir": None,
    "image_root": None,
    "root_dir": "./",
    "folds": [1, 2, 3, 4, 5],
    "fold_id": 1,
    "backbone": "resnet50",
    "pretrained": True,
    "epochs": 20,
    "batch_size": 32,
    "learning_rate": 1e-4,
    "weight_decay": 1e-4,
    "patience": 5,
    "num_workers": 2,
    "use_weighted_sampler": True,
    "use_weighted_loss": False,
    "results_dir": "results",
    "figures_dir": "figures",
    "ckpt_dir": "checkpoints",
}


def load_config(path: str | Path) -> Dict[str, Any]:
    with open(path) as f:
        user = yaml.safe_load(f) or {}
    cfg = {**_CONFIG_DEFAULTS, **user}
    if cfg.get("fold_dir") is None:
        cfg["fold_dir"] = cfg["data_root"]
    if cfg.get("image_root") is None:
        cfg["image_root"] = user.get("image_root")
    if "folds" not in user and "fold_id" in user:
        cfg["folds"] = [user["fold_id"]]
    return cfg


def set_seed(seed: int = SEED, deterministic: bool = True) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def get_device() -> torch.device:
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def setup_logging(log_path: str | Path, name: str = "pcmmd") -> logging.Logger:
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.handlers.clear()
    fmt = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s", "%H:%M:%S")
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    fh = logging.FileHandler(log_path)
    fh.setFormatter(fmt)
    logger.addHandler(sh)
    logger.addHandler(fh)
    logger.propagate = False
    return logger


def save_config_snapshot(cfg: Dict[str, Any], out_path: str | Path) -> None:
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    snap = dict(cfg)
    snap["_torch_version"] = torch.__version__
    snap["_cuda"] = torch.cuda.is_available()
    snap["_seed"] = SEED
    with open(out_path, "w") as f:
        json.dump(snap, f, indent=2, default=str)


def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def model_size_mb(model: nn.Module) -> float:
    n_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
    n_bytes += sum(b.numel() * b.element_size() for b in model.buffers())
    return n_bytes / (1024 ** 2)


@torch.no_grad()
def measure_latency(
    model: nn.Module,
    device: torch.device,
    input_size: tuple = (1, 3, 224, 224),
    n_warmup: int = 5,
    n_runs: int = 30,
) -> Dict[str, float]:
    model.eval().to(device)
    x = torch.randn(*input_size, device=device)
    for _ in range(n_warmup):
        model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model(x)
        if device.type == "cuda":
            torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1000.0)
    arr = np.asarray(times)
    return {"latency_ms_mean": float(arr.mean()), "latency_ms_std": float(arr.std()),
            "throughput_img_s": float(1000.0 / arr.mean())}


def compute_flops(model: nn.Module, input_size: tuple = (1, 3, 224, 224)) -> Dict[str, Optional[float]]:
    try:
        from thop import profile  # type: ignore
        device = next(model.parameters()).device
        x = torch.randn(*input_size, device=device)
        macs, params = profile(model, inputs=(x,), verbose=False)
        return {"macs_g": macs / 1e9, "flops_g": 2 * macs / 1e9, "params_m": params / 1e6}
    except Exception:
        return {"macs_g": None, "flops_g": None, "params_m": None}
    finally:
        for m in model.modules():
            for attr in ("total_ops", "total_params"):
                if hasattr(m, attr):
                    delattr(m, attr)
            m._forward_hooks.clear()
            m._forward_pre_hooks.clear()


def model_complexity_report(
    model: nn.Module,
    device: torch.device,
    input_size: tuple = (1, 3, 224, 224),
) -> Dict[str, Any]:
    rep: Dict[str, Any] = {
        "params_total": count_parameters(model),
        "params_total_m": round(count_parameters(model) / 1e6, 3),
        "model_size_mb": round(model_size_mb(model), 3),
    }
    rep.update({k: (round(v, 3) if isinstance(v, float) else v)
                for k, v in compute_flops(model, input_size).items()})
    rep.update({k: round(v, 3) for k, v in measure_latency(model, device, input_size).items()})
    return rep

Overwriting utils/common.py


In [11]:
!grep -n "device = next" utils/common.py

148:        device = next(model.parameters()).device


In [23]:
!python train_centralized.py --backbone efficientnet_b0 --folds 1

15:56:24 | INFO | Centralized CV | backbone=efficientnet_b0 folds=[1] loss=weighted_ce ft=full | device=cuda
15:56:24 | INFO | Fold 1: train=1624 test=402 | clients=6 val_patients=2 (['2', '7'])
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth
100%|███████████████████████████████████████| 20.5M/20.5M [00:00<00:00, 180MB/s]
15:56:40 | INFO |   [efficientnet_b0 f1] ep 001/30 loss=0.3122 val_f1=0.8807 val_acc=0.9366 val_auc=0.9991
15:56:47 | INFO |   [efficientnet_b0 f1] ep 002/30 loss=0.1010 val_f1=0.9505 val_acc=0.9756 val_auc=0.9997
15:56:54 | INFO |   [efficientnet_b0 f1] ep 003/30 loss=0.0629 val_f1=0.9648 val_acc=0.9829 val_auc=0.9998
15:57:01 | INFO |   [efficientnet_b0 f1] ep 004/30 loss=0.0478 val_f1=0.9896 val_acc=0.9951 val_auc=0.9998
15:57:08 | INFO |   [efficientnet_b0 f1] ep 005/30 loss=0.0408 val_f1=0.9843 val_acc=0.9927 val_auc=0.9997
15:57:15 | INFO |   [e

In [24]:
!python train_kd.py --config configs/kd.yaml

15:58:32 | INFO | KD | teacher=efficientnet_b0 student=mobilenet_v3 fold=1 temps=[2.0, 4.0, 6.0] alpha=0.5 | device=cuda
15:58:38 | INFO | Teacher loaded & frozen: checkpoints/centralized/efficientnet_b0_fold1.pth
15:58:40 | INFO | Training BASELINE student (no teacher)...
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_small-047dcff4.pth
100%|███████████████████████████████████████| 9.83M/9.83M [00:00<00:00, 215MB/s]
15:58:45 | INFO |   [baseline] ep 001/25 loss=0.3079 val_f1=0.7978
15:58:50 | INFO |   [baseline] ep 002/25 loss=0.0870 val_f1=0.9430
15:58:55 | INFO |   [baseline] ep 003/25 loss=0.0894 val_f1=0.9490
15:59:00 | INFO |   [baseline] ep 004/25 loss=0.0477 val_f1=0.9641
15:59:05 | INFO |   [baseline] ep 005/25 loss=0.0309 val_f1=0.9691
15:59:10 | INFO |   [baseline] ep 006/25 loss=0.0241 val_f1=0.9744
15:59:15 | INFO |   [baseline] ep 007/25 loss=0.0310 val_f1=0.9744
15:59:20 | INFO |   [ba